# 실습 1-2 · GPU 사용 확인과 속도 비교

1. 내 PC에 NVIDIA GPU가 있는지 확인
2. 같은 질문을 GPU 강제와 CPU 강제로 실행해 속도 비교
3. 모델이 GPU 메모리에 몇 % 올라갔는지 확인
4. (심화) 내 PC와 MonoRouter(원격)의 응답 속도를 같은 방법으로 비교

GPU/CPU 선택 옵션은 OpenAI 호환 주소(/v1)에서 받지 않아 1~3은 Ollama 고유 API(/api/chat)를 씁니다.

In [2]:
%pip install httpx

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)

   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
   -------------------------- ------------- 2/3 [httpx]
   -------------------------- ------------- 2/3 [httpx]
   -------------------------- ------------- 2/3 [httpx]
   ---------------------------------------- 3/3 [httpx]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import httpx

url = "http://127.0.0.1:11434/api/version"

try:
    print("Ollama 연결 시작...", flush=True)

    response = httpx.get(
        url,
        timeout=5.0,
        trust_env=False
    )

    response.raise_for_status()
    print("연결 성공:", response.json())

except Exception as e:
    print("오류 종류:", type(e).__name__)
    print("오류 내용:", str(e))

Ollama 연결 시작...
연결 성공: {'version': '0.40.3'}


In [5]:
import httpx    # openai를 설치하면 함께 설치되는 HTTP 라이브러리

# OLLAMA = "http://localhost:11434"
# MODEL = "llama3.2:3b"
# GPU_FORCE = {"num_gpu": 99, "main_gpu": 0}   # 모든 층을 첫 번째 GPU에
# CPU_FORCE = {"num_gpu": 0}                    # GPU에 올릴 층 0개

# def ollama_api(path, body=None):
#     if body is None:
#         return httpx.get(OLLAMA + path, timeout=600).json()
#     return httpx.post(OLLAMA + path, json=body, timeout=600).json()

# print("Ollama 버전:", ollama_api("/api/version")["version"])

import httpx

OLLAMA = "http://127.0.0.1:11434"
MODEL = "llama3.2:latest"

GPU_FORCE = {"num_gpu": 99, "main_gpu": 0}
CPU_FORCE = {"num_gpu": 0}

def ollama_api(path, body=None):
    with httpx.Client(
        timeout=600,
        trust_env=False
    ) as client:

        if body is None:
            response = client.get(OLLAMA + path)
        else:
            response = client.post(OLLAMA + path, json=body)

        response.raise_for_status()
        return response.json()

print("Ollama 버전:", ollama_api("/api/version")["version"])



Ollama 버전: 0.40.3


In [6]:
import httpx

for host in ["localhost", "127.0.0.1"]:
    for trust_env in [True, False]:
        try:
            r = httpx.get(
                f"http://{host}:11434/api/version",
                timeout=5,
                trust_env=trust_env
            )
            print(host, trust_env, "성공:", r.status_code)

        except Exception as e:
            print(host, trust_env, "실패:", type(e).__name__)

localhost True 성공: 200
localhost False 성공: 200
127.0.0.1 True 성공: 200
127.0.0.1 False 성공: 200


## 1) 내 PC의 GPU
NVIDIA 드라이버가 있으면 GPU 이름 · 메모리 · 사용률이 나옵니다. 명령을 찾을 수 없다고 나오면 NVIDIA GPU가 없거나 드라이버가 없는 것입니다(실습은 CPU로 진행됨). Apple Silicon Mac은 Ollama가 자동으로 GPU를 씁니다.

In [3]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

name, memory.total [MiB], memory.used [MiB], utilization.gpu [%]
NVIDIA GeForce RTX 3060 Laptop GPU, 6144 MiB, 0 MiB, 0 %


## 2) GPU 강제 vs CPU 강제
조건마다 한 번 예열(모델을 다시 올림)한 뒤 측정합니다. Ollama 응답의 시간 단위는 나노초입니다.

In [7]:
def run_once(options):
    r = ollama_api("/api/chat", {
        "model": MODEL, "stream": False,
        "messages": [{"role": "user", "content": "보안관제에서 오탐을 줄이는 방법 세 가지를 한 줄씩 써 줘."}],
        "options": dict(options, temperature=0, num_predict=128)})       # 출력 길이를 128토큰으로 맞춤
    gen_tps = r["eval_count"] / r["eval_duration"] * 1e9                  # 초당 생성 토큰
    in_tps = r["prompt_eval_count"] / max(r["prompt_eval_duration"], 1) * 1e9   # 초당 입력 처리 토큰
    return gen_tps, in_tps, r["total_duration"] / 1e9

def gpu_share():
    """/api/ps: 메모리에 올라간 모델 중 GPU 메모리에 올라간 비율 (명령 창의 ollama ps와 같음)"""
    
    for m in ollama_api("/api/ps")["models"]:
        if m["name"].startswith(MODEL):
            #size_mib = m["size"] / (1024 * 1024)
            #vram_mib = m["size_vram"] / (1024 * 1024)       
   
            #print(f"전체 모델 크기: {size_mib:.2f} MiB")
            #print(f"VRAM 적재 크기: {vram_mib:.2f} MiB")
            
            return 100 * m["size_vram"] / m["size"]   
            
for label, opts in [("GPU 강제", GPU_FORCE), ("CPU 강제", CPU_FORCE)]:
    run_once(opts)                                   # 예열
    gen, inp, total = run_once(opts)
    print(f"{label}: 생성 {gen:6.1f} 토큰/초 | 입력 처리 {inp:7.1f} 토큰/초 | 전체 {total:5.1f}초 | GPU 적재 {gpu_share():.0f}%")

GPU 강제: 생성  111.8 토큰/초 | 입력 처리  5790.6 토큰/초 | 전체   1.2초 | GPU 적재 100%
CPU 강제: 생성   23.1 토큰/초 | 입력 처리  1049.5 토큰/초 | 전체   5.6초 | GPU 적재 0%


### 읽는 법
- GPU 강제의 GPU 적재가 100%이고 생성 속도가 CPU 강제보다 몇 배 빠르면 정상입니다.
- GPU 강제인데 GPU 적재가 0%면 Ollama가 GPU 라이브러리를 읽지 못하는 상태입니다 → 발표자료 'GPU를 쓰지 않을 때' 장표의 조치.
- GPU가 없는 PC는 두 결과가 비슷한 것이 정상입니다.

In [25]:
ollama_api("/api/chat", {"model": MODEL, "messages": [], "keep_alive": 0})   # 모델을 내려 다음 실행은 기본 설정으로
print("모델을 메모리에서 내림")

모델을 메모리에서 내림


## 3) (심화) 내 PC vs MonoRouter: 같은 방법으로 속도 재기
답을 스트리밍으로 받아 시간을 나눕니다.
- 첫 토큰 시간: 요청 → 첫 글자. 네트워크 왕복 + 입력 처리(+ 원격은 대기열)
- 출력 속도: 첫 글자 → 끝 사이의 초당 토큰. 생성과 전송이 겹쳐 네트워크 영향이 거의 빠짐 → 서버 GPU의 생성 속도에 가까움
- 네트워크 왕복: 모델 목록 요청 3회 중 가장 빠른 값

원격 서버의 GPU 종류 · 개수는 API로 알 수 없습니다. MonoRouter 키를 넣어야 원격 측정이 됩니다.

In [27]:
import statistics, time
from openai import OpenAI

MONO_KEY = "mr_openai_e0di9bdw3iFdUDG1hgAxEfo_tH_DXI2z"
TARGETS = [("내 PC (Ollama)", OpenAI(base_url=OLLAMA + "/v1", api_key="ollama"), MODEL),
           ("원격 (MonoRouter)", OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1", api_key=MONO_KEY), "gpt-5.5")]
PROMPT = "다음 보안 로그에서 위험해 보이는 줄을 골라 한 줄씩 설명해 줘.\n" + open("sample_logs.txt", encoding="utf-8").read() * 3

def stream_once(client, model):
    t0 = time.perf_counter()
    stream = client.chat.completions.create(model=model, stream=True, stream_options={"include_usage": True},
                                            messages=[{"role": "user", "content": PROMPT}])
    first, usage = None, None
    for ch in stream:
        if ch.usage:
            usage = ch.usage                           # 마지막 조각에 토큰 수가 옴
        if ch.choices and ch.choices[0].delta.content and first is None:
            first = time.perf_counter()
    end = time.perf_counter()
    return first - t0, (usage.completion_tokens - 1) / (end - first), end - t0, usage.prompt_tokens

def network_rtt(client):
    """가벼운 요청(모델 목록) 3번 중 가장 빠른 왕복 시간"""
    times = []
    for _ in range(3):
        t = time.perf_counter()
        client.models.list()
        times.append(time.perf_counter() - t)
    return min(times)

for name, client, model in TARGETS:
    try:
        rtt = network_rtt(client)
        stream_once(client, model)                     # 예열
        runs = [stream_once(client, model) for _ in range(3)]
        ttft, out_tps, total, in_tok = [statistics.median(r[i] for r in runs) for i in range(4)]
        print(f"{name:<15} 입력 {in_tok:.0f}토큰 | 왕복 {rtt:.3f}초 | 첫 토큰 {ttft:.2f}초 "
              f"| 입력 처리(추정) {in_tok / max(ttft - rtt, 1e-3):.0f} 토큰/초 | 출력 {out_tps:.1f} 토큰/초 | 전체 {total:.1f}초")

        log_msg = (
        f"{name:<15} 입력 {in_tok:.0f}토큰 | 왕복 {rtt:.3f}초 | 첫 토큰 {ttft:.2f}초 "
        f"| 입력 처리(추정) {in_tok / max(ttft - rtt, 1e-3):.0f} 토큰/초 "
        f"| 출력 {out_tps:.1f} 토큰/초 | 전체 {total:.1f}초"
        )
        # 화면 출력도 같이 하고 싶다면 그대로 print
        #print(log_msg)

    except Exception as e:
        print(f"{name:<15} 측정 실패: {type(e).__name__} (키 · 연결 확인)")

내 PC (Ollama)   입력 3352토큰 | 왕복 0.006초 | 첫 토큰 3.17초 | 입력 처리(추정) 1059 토큰/초 | 출력 98.7 토큰/초 | 전체 13.6초
내 PC (Ollama)   입력 3352토큰 | 왕복 0.006초 | 첫 토큰 3.17초 | 입력 처리(추정) 1059 토큰/초 | 출력 98.7 토큰/초 | 전체 13.6초
MonoRouter      입력 3331토큰 | 왕복 0.106초 | 첫 토큰 8.31초 | 입력 처리(추정) 406 토큰/초 | 출력 161.7 토큰/초 | 전체 18.0초
MonoRouter      입력 3331토큰 | 왕복 0.106초 | 첫 토큰 8.31초 | 입력 처리(추정) 406 토큰/초 | 출력 161.7 토큰/초 | 전체 18.0초


- 원격 값은 시간대와 동시 사용자 수에 따라 달라집니다. 같은 시간에 두세 번 재서 봅니다.
- 추론형 원격 모델은 보이지 않는 추론 토큰을 먼저 만들어 첫 토큰 시간이 길어지고, 출력 속도는 실제보다 낮게 계산될 수 있습니다.

In [30]:
prompt = f"""
다음은 로컬 LLM 성능(처리 속도) 테스트 결과 데이터야.
{log_msg}

이 결과를 바탕으로 현재 로컬 LLM 이용과 원격 LLM 이용을 SOC 분석가 관점에서 간략하게 비교, 진단해 줘. 3줄이내로.
"""

# 앞서 사용하던 ollama_api를 통해 질의하는 예시
response = ollama_api("/api/chat", {
    "model": MODEL, 
    "stream": False,
    "messages": [
        {"role": "system", "content": "당신은 전문 성능 분석가입니다."},
        {"role": "user", "content": prompt}
    ]
})

# LLM의 답변 출력
print("\n--- LLM 성능 분석 결과 ---")
print(response["message"]["content"])


--- LLM 성능 분석 결과 ---
현재 로컬 LLM 이용의 장점은 대량의 데이터 처리(406 토큰/초)과 속도(0.106초)가 적극적으로 보임을 볼 수 있습니다. 반면 원격 LLM 이용의 단점은 느린 처리 속도(161.7 토큰/초)와 초기 Token 처리 시간(8.31초)가 나타났습니다.
